In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Modules and Model Construction

The multilayer perceptron of that section had four
parameter arrays, and we could name each one by hand. The models in the rest
of this book have many more. A residual network for images stacks dozens of
layers, and the language model Llama 3 8B
[@Grattafiori.Dubey.Jauhri.ea.2024], with 8 billion parameters,
consists of 32 identical blocks and stores its parameters in 291 arrays.
Every one of them must reach the optimizer, be written to a checkpoint, and
move to the GPU. An array that is left out raises no error: it is simply
never trained, or never saved.

Deep learning frameworks solve this with one abstraction, the *module*. A
module is an object that owns parameters, owns other modules (its
*children*), and defines a forward computation from inputs to outputs.
Layers, groups of layers, and whole models are all modules, so a model is a
tree of modules, as in the figure. The framework finds every
parameter by walking this tree, provided that each child has been
*registered* with its parent. This section is a practical one, and a reader
who has written modules before will move through much of it quickly. It
shows how registration works and how it fails, which is the part not to
skip, how to write the forward computation of a module ourselves, and how
to describe a whole model by a handful of numbers. The rest of the chapter uses the same walk for other tasks: counting
the parameters (that section), initializing them
(that section), saving them (that section), and
moving them to a GPU (that section).

![The module tree of the residual MLP that we build at the end of this section. Each edge carries the name under which the parent registered the child, and the path of names from the root to a parameter is the name of that parameter, here `1.body.0.weight`. Blocks 2 to 4 repeat block 1.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-module-names.svg)

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

In [ ]:
from dataclasses import dataclass

## The Module Abstraction

### Sequential Containers and the Registry

Every model since that section has been a module: `d2l.Module`
is a subclass of the framework's module class. The quickest way to build a
module from layers is a *sequential container*, a module that applies its
children one after the other. Here it holds the MLP of
that section, with a hidden layer of 256 ReLU units and
10 outputs, applied to a minibatch of two examples with 20 features each.

In Flax NNX the module class is `nnx.Module`, and the container
`nnx.Sequential` takes its layers as arguments. NNX layers create their
parameters in the constructor, so each `nnx.Linear` needs its number of
inputs and `rngs`, the source of random keys for its initial weights.

In [ ]:
rngs = nnx.Rngs(d2l.get_key())
net = nnx.Sequential(nnx.Linear(20, 256, rngs=rngs), nnx.relu,
                     nnx.Linear(256, 10, rngs=rngs))

X = jax.random.uniform(d2l.get_key(), (2, 20))
net(X).shape

The container keeps its children in a *registry*, a table from names to
modules, and its forward computation runs through that table in order.

In NNX the module and its attributes form a graph of Python objects, and
`nnx.state(net, nnx.Param)` lists every parameter by its path through that
graph. The container keeps its children under `layers`, at the positions 0,
1, and 2; the ReLU at position 1 is a plain function and owns nothing.

In [ ]:
[(path, value.shape)
 for path, value in nnx.state(net, nnx.Param).flat_state()]

This table is where every walk over the model starts. Collecting the
parameters for the optimizer, writing them to a file, and moving them to a
GPU all visit the registered children, then their children, and so on down
to the parameters. For all three, a layer that is not in the table does not
exist.

### Subclasses of the Module Class

A sequential container only expresses a chain. For any other wiring we
subclass the module class and write two methods: a constructor that creates
the children, and a forward method that applies them to the input. The
forward method is called `forward` in PyTorch and Gluon, `call` in Keras,
and `__call__` in NNX.

The `d2l.Module` of earlier chapters defines `__call__` to run `forward`,
which is why its subclasses wrote `forward`; a plain NNX module implements
`__call__` directly.

Here is the same MLP once more:

In [ ]:
class MLP(nnx.Module):
    def __init__(self, rngs):
        self.hidden = nnx.Linear(20, 256, rngs=rngs)
        self.out = nnx.Linear(256, 10, rngs=rngs)

    def __call__(self, X):
        return self.out(nnx.relu(self.hidden(X)))

In [ ]:
net = MLP(nnx.Rngs(d2l.get_key()))
net(X).shape

Two details make this work. First, `self.hidden = ...` is not an ordinary
attribute assignment. The module class intercepts assignment (the Python
method `__setattr__`), and when the value is a module it enters the
registry under the attribute name, here `hidden` and `out`. That is all it
takes for both layers' parameters to reach the optimizer. Second, we wrote
no backward method: automatic differentiation derives the gradients from
whatever the forward method computes.

In NNX, `net(X)` runs `__call__` directly; the model needs no extra
dictionary of parameters, because the parameters are attributes of the
layers. The transformations `nnx.grad` and `nnx.jit` follow the same
attributes.

## Module Containers

### Writing a Container

A container has two jobs: to register each child, and to run the children
in order. It is instructive to write one ourselves, since the first job is
the one that can go wrong.

In [ ]:
class MySequential(nnx.Module):
    def __init__(self, *modules):
        self.modules = nnx.List(modules)

    def __call__(self, X):
        for module in self.modules:
            X = module(X)
        return X

`nnx.List` is a list that NNX treats as part of the module graph, so the
parameters of both `Linear` layers appear under the path `modules`.

Our container works like the built-in one:

In [ ]:
rngs = nnx.Rngs(d2l.get_key())
net = MySequential(nnx.Linear(20, 256, rngs=rngs), nnx.relu,
                   nnx.Linear(256, 10, rngs=rngs))
net(X).shape

### Layers in a Plain List

Now suppose that we store the layers in an ordinary Python list, as anyone
would who had not read the previous subsection. The forward method can
still loop over the list, but the list itself is not a module. Does the
framework still find the layers inside it?

In [ ]:
class PlainListMLP(nnx.Module):
    def __init__(self, rngs):
        self.layers = [nnx.Linear(20, 256, rngs=rngs),
                       nnx.Linear(256, 10, rngs=rngs)]  # Plain list

    def __call__(self, X):
        return self.layers[1](nnx.relu(self.layers[0](X)))

try:
    net = PlainListMLP(nnx.Rngs(d2l.get_key()))
except ValueError as e:
    print(str(e).split(' of Pytree')[0])  # The rest lists the repairs

NNX rejects the plain list when the model is constructed. It would
otherwise treat the list as fixed configuration and hide the parameters
inside it. The first line of the error, which is all we print, names the
problem, and the rest of the message lists the repairs, among them
`nnx.List` and `nnx.data`. The same error stops a model that registers its
outer layers as attributes and keeps only the layers between them in a
plain list, so a partly registered model cannot be built. With the
graph-aware `nnx.List`, the model is complete:

In [ ]:
class ListMLP(PlainListMLP):
    def __init__(self, rngs):
        self.layers = nnx.List([nnx.Linear(20, 256, rngs=rngs),
                                nnx.Linear(256, 10, rngs=rngs)])

net = ListMLP(nnx.Rngs(d2l.get_key()))
state = nnx.state(net, nnx.Param)
net(X).shape, sum(x.size for _, x in state.flat_state())

All 7,946 parameters are present: 20 · 256 + 256 in the hidden layer and
256 · 10 + 10 in the output layer. NNX supplies `nnx.List` and `nnx.Dict`
for collections of modules. A misdeclared child fails when the model is
constructed, not after a training run that updated nothing.

## Forward Computation

The forward method is ordinary Python. It can call any tensor operation,
loop, branch, and combine intermediate results. We use this freedom for a
residual connection, and then ask what a compiled training step takes away
from it.

### Residual Connections

Let us start with the simplest module whose forward method is not a chain.
A *residual connection* adds the input of a block to its output
[@He.Zhang.Ren.ea.2016*1]. If $f$ is the *body* of the block, the
computation inside it, here two fully connected layers with a ReLU between
them, the block computes

$$\mathbf{y} = \mathbf{x} + f(\mathbf{x}).$$

If $f$ is zero, the block is the identity, so a stack of such blocks can
pass its input through unchanged, and each block only has to learn a
correction $f(\mathbf{x})$ to its input. The networks of
that section are built this way. No framework provides the residual
connection as a layer, and none needs to. It is one line of the forward
method, `X + self.body(X)`:

In [ ]:
class ResidualBlock(nnx.Module):
    def __init__(self, num_hiddens, *, rngs):
        self.body = nnx.Sequential(
            nnx.Linear(num_hiddens, num_hiddens, rngs=rngs), nnx.relu,
            nnx.Linear(num_hiddens, num_hiddens, rngs=rngs))

    def __call__(self, X):
        return X + self.body(X)

![A residual block. The input splits into the body, Linear, ReLU, Linear, and an identity path, and the two rejoin by addition.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-residual-block.svg)

the figure shows the wiring. The addition has one
consequence for construction: $\mathbf{x}$ and $f(\mathbf{x})$ must have
the same shape, so a residual block has a single width, which we pass to
the constructor.

In [ ]:
block = ResidualBlock(24, rngs=nnx.Rngs(d2l.get_key()))
X24 = jax.random.normal(d2l.get_key(), (2, 24))
block(X24).shape

A forward method may also read values that are neither inputs nor
parameters. A fixed number can stay a plain Python attribute. An array
computed from data, such as the mean of the inputs, is another matter: it
must be saved and moved with the model, yet never trained.
that section shows how to register such state.

### Eager and Compiled Execution

So far the forward method has also run like ordinary Python, one line at a
time. A framework can instead record the computation once and replay the
recording. This is faster, but it restricts what the forward method may do.

Called directly, `__call__` runs as written, one operation at a time, a
mode called *eager execution*. The `Trainer` of that section
compiles each training step with `nnx.jit` instead: it runs the step once
with *tracers*, placeholder arrays that have shapes but no values, records
the operations, and from then on executes the recording. A loop whose
condition depends on the values of arrays cannot be recorded this way:

In [ ]:
class HalvingNet(nnx.Module):
    def __init__(self, rngs):
        self.proj = nnx.Linear(20, 24, rngs=rngs)

    def __call__(self, X):
        X = self.proj(X)
        while jnp.abs(X).sum() > 1:  # Depends on the values of X
            X = X / 2
        return X.sum()

net = HalvingNet(nnx.Rngs(d2l.get_key()))
print(net(X))
try:
    nnx.jit(lambda net, X: net(X))(net, X)
except jax.errors.TracerBoolConversionError as e:
    print(type(e).__name__)

Eager execution runs the loop and returns a number. Under `nnx.jit` the
condition is a tracer without a value, and JAX refuses to turn it into a
Python `bool`. A loop that depends on values must be written with
`jax.lax.while_loop`, which becomes part of the compiled program. Loops over
the structure of the model, such as the loop over the children in
`MySequential`, cause no trouble: they run once while the step is traced.

## Lazy Initialization

### Shapes from Data

NNX makes a different choice from the other three frameworks, whose layers
can leave the input width open until the first minibatch arrives. An
`nnx.Linear` takes both widths and creates its parameters in the
constructor, so the model is complete at once: an optimizer or a checkpoint
can see every parameter without a forward pass, and the shapes can be read
as soon as construction returns:

In [ ]:
rngs = nnx.Rngs(d2l.get_key())
net = nnx.Sequential(nnx.Linear(20, 256, rngs=rngs), nnx.relu,
                     nnx.Linear(256, 10, rngs=rngs))
[(path, value.shape)
 for path, value in nnx.state(net, nnx.Param).flat_state()]

### Initialization after a Dry Run

Lazy layers spare us the arithmetic of input widths, which in a deep model
is tedious and easy to get wrong. Unfortunately, the parameters then do not
exist until a minibatch has passed through. Before that, they cannot be
counted, and an initializer whose scale depends on the number of inputs,
such as the Xavier initialization of that section, cannot be
applied. Initialization therefore follows a *dry run*, one forward pass on a
representative minibatch whose output is discarded.

The initializer is an argument of the layer: `nnx.Linear` accepts a
`kernel_init` function and calls it in the constructor. Xavier's uniform
initializer is `nnx.initializers.xavier_uniform()`:

In [ ]:
class TinyMLP(d2l.Module):
    def __init__(self, rngs):
        super().__init__()
        self.net = nnx.Sequential(
            nnx.Linear(20, 256,
                       kernel_init=nnx.initializers.xavier_uniform(),
                       rngs=rngs), nnx.relu,
            nnx.Linear(256, 10, rngs=rngs))

model = TinyMLP(nnx.Rngs(d2l.get_key()))
[(path, value.shape)
 for path, value in nnx.state(model, nnx.Param).flat_state()]

The constructor created every parameter and drew the first kernel from the
Xavier initializer.

Which should we prefer? For a quick experiment on new data, lazy layers are
a convenience. A model whose parameters are to be counted, initialized by a
rule, or built from a config should state both widths of every layer, as
the models in the rest of this chapter do.

## Models from a Config

### Configs and Builders

Could we rebuild a model from its saved weights alone? Not quite. A trained
model is a set of arrays plus the code that knows what to do with them.
Every width and depth in this section so far was a literal typed into a
constructor call, and a checkpoint of the weights records such choices at
best indirectly: the shapes of the arrays reveal the widths, but nothing in
them reveals a choice without parameters, such as the activation function.

The choices that vary from one experiment to the next therefore belong in a
small *configuration* object, or *config*, that is logged with the results
and saved with the weights. The topology, meaning what a block computes and
how blocks connect, stays in code. Here a Python `dataclass` holds four
fields, and `build` turns them into a model:

In [ ]:
@dataclass
class MLPConfig:
    num_inputs: int = 784
    num_hiddens: int = 256
    num_blocks: int = 4
    num_outputs: int = 10

def build(cfg: MLPConfig, rngs=None) -> nnx.Module:
    rngs = nnx.Rngs(d2l.get_key()) if rngs is None else rngs
    blocks = [ResidualBlock(cfg.num_hiddens, rngs=rngs)
              for _ in range(cfg.num_blocks)]
    return nnx.Sequential(
        nnx.Linear(cfg.num_inputs, cfg.num_hiddens, rngs=rngs), *blocks,
        nnx.Linear(cfg.num_hiddens, cfg.num_outputs, rngs=rngs))

One config gives one architecture: an input layer from `num_inputs`
features to the hidden width, `num_blocks` residual blocks, and an output
layer. The blocks pass through a plain Python list, of all things. This is
safe here, because unpacking the list into the sequential container
registers each block. Every layer receives both of its widths, so the model
needs no dry run before its parameters can be counted.

The paths of the first parameters are those of the figure. NNX
lists each path as a tuple, and a sequential container keeps its children
under `layers`:

In [ ]:
net = build(MLPConfig())
[(path, value.shape)
 for path, value in nnx.state(net, nnx.Param).flat_state()][:4]

The model maps a minibatch of 784-dimensional inputs, the size of a
flattened Fashion-MNIST image, to 10 outputs:

In [ ]:
net(jax.random.uniform(d2l.get_key(), (2, 784))).shape

### Counting Parameters

The config also tells us the size of the model before we build it. The
input layer has $784 \cdot 256 + 256 = 200{,}960$ parameters, each block
has two layers of $256 \cdot 256 + 256$, or $131{,}584$ together, and the
output layer has $256 \cdot 10 + 10 = 2{,}570$. The default model therefore
has

$$200{,}960 + 4 \cdot 131{,}584 + 2{,}570 = 729{,}866$$

parameters, most of them in the blocks. Rescaling the model is a change to
two fields, not an edit of the model code, and a walk over the parameters
confirms the count:

In [ ]:
for cfg in (MLPConfig(), MLPConfig(num_hiddens=512, num_blocks=8)):
    n = sum(x.size for _, x in nnx.state(build(cfg), nnx.Param).flat_state())
    print(f'num_hiddens={cfg.num_hiddens}, num_blocks={cfg.num_blocks}: '
          f'{n:,} parameters')

Doubling the width multiplies the parameters of each block by about four,
and doubling the number of blocks doubles their total, so the larger
config has more than six times as many parameters. The 8B model of the
introduction is described the same way: a config of a dozen numbers, such
as 32 blocks and a width of 4,096, and code that turns it into a module
tree.

Note, however, that a config records numbers, not code. Rebuilding a model
from a checkpoint needs the weights, the config, and the same `build`
function and block classes. that section saves the first two
together in one file; the code we must keep ourselves.


Throughout this section the parameters were things to be found and
counted. We have not yet looked at them: which tensors a model holds, how
much memory they take, and how to share or freeze them. This is the subject
of that section.

## Summary

Three things are worth remembering. A model is a tree of modules, and the
name of each parameter is its path from the root. The framework finds
parameters by walking the registered children, so a child stored where the
walk cannot see it is never trained or saved, and in the worst case nothing
tells us. A config of a few numbers, together with a `build` function,
records an architecture so that it can be logged, saved, and rebuilt. The
rest is technique: a forward method written in ordinary Python, which a
compiled training step restricts, and lazy layers, which must see a
minibatch before they can be initialized.

In NNX, the attributes of a module form its graph. Collections of modules
go into `nnx.List` or `nnx.Dict`; a plain list of modules is rejected when
the model is constructed.

## Exercises

1. [code] **Unregistered containers.** NNX rejects a plain Python list that
   holds modules and names `nnx.List` and `nnx.data` as repairs.
    1. Rewrite `ListMLP` twice: once with its two layers in an `nnx.Dict`
       keyed `'hidden'` and `'out'`, and once with the plain list wrapped
       in `nnx.data`. Verify that all 7,946 parameters appear in
       `nnx.state(net, nnx.Param)` in both versions.
    1. Compare the parameter paths of the two versions with those of
       `ListMLP`. A checkpoint stores the state as a mapping from paths to
       arrays. Which of the two rewrites could restore a checkpoint saved
       from `ListMLP`, and why?

2. **Widening a residual block.** `ResidualBlock` requires its input and
   output widths to agree. Suppose a block maps width $d$ to a larger width
   $d'$, so its body ends in a $d'$-wide layer and the skip path must also
   produce width $d'$. Give one way to build that skip path with
   parameters and one without. For each, count the extra parameters and
   state which output coordinates still receive the identity path.

3. [code] **Config boundaries.** Extend `MLPConfig` with a field
   `act: str = 'relu'` that selects the activation inside every
   `ResidualBlock`, and make `build` honor it.
    1. Build one model with `act='tanh'` and one from the default config,
       and compare their parameter shapes. Could a checkpoint that holds
       only the weights tell the two models apart? State what the answer
       implies for the config.
    1. A plain feed-forward block has the same `body` as `ResidualBlock`
       but no skip connection. Apply the same test to the choice between
       the two block types, and decide whether that choice belongs in the
       config or in code.

4. [code] **A module from scratch.** ● In plain Python and NumPy, without any
   framework base class, write a `Parameter` class that wraps an array and
   a minimal `Module` class whose `__setattr__` registers every assigned
   `Parameter` or `Module`. Give `Module` a `parameters()` method that
   recurses into the registered children. Write a `Linear` subclass of
   `Module` that holds a weight and a bias, build a two-layer MLP from two
   instances of it, and verify that `parameters()` returns all four arrays.
   Then store the two layers in a plain list, check what `parameters()`
   returns, and extend `Module` so that the list version finds all four
   arrays as well.

    *Adapted from CMU 10-714,
    [Homework 2](https://github.com/dlsyscourse/hw2), Question 2.*

5. [code] **Shared children.** Reusing one child module at several sites
   turns the module tree of the figure into a graph. Build a
   small network that uses the same child at two sites. Predict, then
   verify: does the shared child's parameter set appear once or twice in
   `nnx.state(net, nnx.Param)`? The function `nnx.split(net)` separates a
   module into a static graph definition and its state, and `nnx.merge`
   rebuilds a module from the two. After such a round trip, are the two
   sites still one object, checked with `is`?

[Discussions](https://d2l.discourse.group/t/36752)